# 02 · Data Cleaning: NYC Capital Projects

**Project:** Late and Over Budget: What Drives Delays and Cost Overruns in NYC's Capital Projects?

This notebook cleans the three raw datasets from NYC Open Data's Capital Projects Dashboard and saves analysis-ready tables to `data/processed/` for loading into SQL.

| Raw table | Rows | Grain |
|---|---|---|
| budget_schedule | 56,525 | One row per FMS ID per reporting snapshot (10 snapshots, May 2023 – May 2026) |
| schedule_history | 22,464 | One row per project (PID) per reported completion date |
| budget_spend_history | 53,495 | One row per FMS ID per reported year-month |

**Key issues found during profiling (`01_data_profiling`):**
1. All date columns are stored as text.
2. `PID` is missing on ~47% of budget_schedule rows (budget lines without an agency schedule), which also forces it to float.
3. `Current Phase` has ~60 inconsistent labels (case, hyphens, parentheses, synonyms).
4. Delay reasons contain near-duplicate labels (e.g. "HAZARDOUS CONDITION" vs "HAZARDOUS CONDITIONS").
5. Agencies are only abbreviations.

Every fix is recorded in a data-quality log saved alongside the outputs.

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, re
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/nyc-capital-projects-analysis'
RAW = f'{BASE}/data/raw'
PROCESSED = f'{BASE}/data/processed'
os.makedirs(PROCESSED, exist_ok=True)

pd.set_option('display.max_columns', 50)

bs_raw  = pd.read_csv(f'{RAW}/budget_schedule.csv', low_memory=False)
sh_raw  = pd.read_csv(f'{RAW}/schedule_history.csv', low_memory=False)
bsh_raw = pd.read_csv(f'{RAW}/budget_spend_history.csv', low_memory=False)

for name, df in [('budget_schedule', bs_raw), ('schedule_history', sh_raw), ('budget_spend_history', bsh_raw)]:
    print(f'{name:22s} {df.shape[0]:>7,} rows  {df.shape[1]:>3} cols')

budget_schedule         56,525 rows   27 cols
schedule_history        22,464 rows   10 cols
budget_spend_history    53,495 rows    8 cols


## 2 · Helper functions
A small data-quality log records every change so the cleaning is transparent and reproducible.

In [ ]:
dq_log = []

def log(table, step, detail, rows_affected=None):
    dq_log.append({'table': table, 'step': step, 'detail': detail, 'rows_affected': rows_affected})
    extra = f' ({rows_affected:,} rows)' if rows_affected is not None else ''
    print(f'[{table}] {step}: {detail}{extra}')

def snake_case(col):
    col = col.strip().lower()
    col = col.replace('(%)', 'pct').replace('%', 'pct')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return col.strip('_')

def strip_text(df, table):
    obj_cols = df.select_dtypes(include='object').columns
    for c in obj_cols:
        df[c] = df[c].str.strip()
        df[c] = df[c].replace('', np.nan)
    log(table, 'Trim text', f'Stripped whitespace in {len(obj_cols)} text columns; blanks set to null')
    return df

def parse_dates(df, cols, table, min_year=1980, max_year=2050):
    for c in cols:
        before = df[c].notna().sum()
        df[c] = pd.to_datetime(df[c], errors='coerce', format='mixed')
        failed = before - df[c].notna().sum()
        out_of_range = df[c].notna() & ((df[c].dt.year < min_year) | (df[c].dt.year > max_year))
        n_oor = int(out_of_range.sum())
        df.loc[out_of_range, c] = pd.NaT
        if failed or n_oor:
            log(table, 'Date issues', f'{c}: {failed} unparseable, {n_oor} outside {min_year}-{max_year} set to null', failed + n_oor)
    log(table, 'Parse dates', f'Converted {len(cols)} text columns to datetime')
    return df

def yyyymm_to_date(series):
    return pd.to_datetime(series.astype('Int64').astype(str), format='%Y%m', errors='coerce')

def drop_exact_dupes(df, table):
    n = df.duplicated().sum()
    if n:
        df = df.drop_duplicates()
    log(table, 'Exact duplicates', 'Removed fully duplicated rows', int(n))
    return df

## 3 · Clean `budget_schedule`

In [ ]:
bs = bs_raw.copy()
T = 'budget_schedule'

# 3.1 Column names → snake_case
bs.columns = [snake_case(c) for c in bs.columns]
log(T, 'Rename columns', 'Converted column names to snake_case')
print(list(bs.columns))

[budget_schedule] Rename columns: Converted column names to snake_case
['reporting_period', 'managing_agency', 'sponsor_agency', 'pid', 'fms_id', 'total_budget', 'spend_to_date', 'spend_to_date_pct', 'fms_project_name', 'agency_project_name', 'agency_project_description', 'current_phase', 'current_phase_start', 'forecast_current_phase_end', 'forecast_completion', 'actual_design_start', 'actual_design_end', 'actual_construction_procurement_start', 'actual_construction_procurement_end', 'actual_construction_start', 'actual_construction_end', 'borough', 'community_board', 'budget_line', 'ten_year_plan_category', 'agency_data_date', 'fms_data_date']


In [ ]:
# 3.2 Text, duplicates, keys
bs = strip_text(bs, T)
bs = drop_exact_dupes(bs, T)

bs['reporting_period'] = yyyymm_to_date(bs['reporting_period'])
bs['fms_id'] = bs['fms_id'].str.upper()
bs['pid'] = bs['pid'].astype('Int64')          # nullable integer: keeps missing PIDs as <NA>
bs['has_schedule'] = bs['pid'].notna()
log(T, 'Keys', 'reporting_period → date; fms_id upper-cased; pid → nullable integer', int((~bs['has_schedule']).sum()))

[budget_schedule] Trim text: Stripped whitespace in 22 text columns; blanks set to null
[budget_schedule] Exact duplicates: Removed fully duplicated rows (0 rows)
[budget_schedule] Keys: reporting_period → date; fms_id upper-cased; pid → nullable integer (26,461 rows)


In [ ]:
# 3.3 Dates
date_cols_bs = [
    'current_phase_start', 'forecast_current_phase_end', 'forecast_completion',
    'actual_design_start', 'actual_design_end',
    'actual_construction_procurement_start', 'actual_construction_procurement_end',
    'actual_construction_start', 'actual_construction_end',
    'agency_data_date', 'fms_data_date',
]
date_cols_bs = [c for c in date_cols_bs if c in bs.columns]
bs = parse_dates(bs, date_cols_bs, T)

[budget_schedule] Date issues: current_phase_start: 0 unparseable, 2 outside 1980-2050 set to null (2 rows)
[budget_schedule] Date issues: forecast_completion: 1 unparseable, 0 outside 1980-2050 set to null (1 rows)
[budget_schedule] Date issues: actual_design_start: 0 unparseable, 2 outside 1980-2050 set to null (2 rows)
[budget_schedule] Date issues: actual_design_end: 0 unparseable, 3 outside 1980-2050 set to null (3 rows)
[budget_schedule] Date issues: actual_construction_end: 0 unparseable, 1 outside 1980-2050 set to null (1 rows)
[budget_schedule] Parse dates: Converted 11 text columns to datetime


### 3.4 Standardise `current_phase`
~60 raw labels collapse into a clean phase name and a broader **phase group** used in the dashboard. The raw value is kept for traceability, and `phase_in_brackets` records whether the label was wrapped in parentheses (checked against `has_schedule` below).

In [ ]:
PHASE_MAP = {
    'pending': 'Pending',
    'pre design': 'Pre-Design', 'initiation': 'Initiation', 'scope development': 'Scope Development',
    'design': 'Design', 'design build': 'Design-Build', 'design built': 'Design-Build',
    'construction procurement': 'Construction Procurement', 'pre construction phase': 'Pre-Construction',
    'construction': 'Construction',
    'close out': 'Close-out', 'closeout': 'Close-out',
    'completed': 'Completed',
    'on hold': 'On Hold', 'inactive': 'Inactive',
    'cancelled': 'Cancelled', 'terminated': 'Terminated', 'withdrawn': 'Withdrawn',
    'rescinded': 'Rescinded', 'defunded': 'Defunded', 'funding withdrawn': 'Funding Withdrawn',
    'defaulted': 'Defaulted', 'not fully funded': 'Not Fully Funded', 'capitally ineligible': 'Capitally Ineligible',
    'n/a': np.nan,
}

PHASE_GROUP = {
    'Pre-Design': '1 Planning', 'Initiation': '1 Planning', 'Scope Development': '1 Planning',
    'Design': '2 Design', 'Design-Build': '2 Design',
    'Construction Procurement': '3 Procurement', 'Pre-Construction': '3 Procurement',
    'Construction': '4 Construction',
    'Close-out': '5 Close-out',
    'Completed': '6 Completed',
    'On Hold': 'On Hold / Inactive', 'Inactive': 'On Hold / Inactive',
    'Cancelled': 'Stopped', 'Terminated': 'Stopped', 'Withdrawn': 'Stopped', 'Rescinded': 'Stopped',
    'Defunded': 'Stopped', 'Funding Withdrawn': 'Stopped', 'Defaulted': 'Stopped',
    'Not Fully Funded': 'Stopped', 'Capitally Ineligible': 'Stopped',
    'Pending': 'Pending',
}

def normalise_phase(x):
    if pd.isna(x):
        return np.nan
    key = re.sub(r'[()]', '', str(x)).replace('-', ' ').lower()
    key = re.sub(r'\s+', ' ', key).strip()
    return PHASE_MAP.get(key, key.title())   # unmapped labels (contract/funding types) → Title Case

bs['current_phase_raw'] = bs['current_phase']
bs['phase_in_brackets'] = bs['current_phase'].str.startswith('(', na=False)
bs['current_phase'] = bs['current_phase_raw'].apply(normalise_phase)
bs['phase_group'] = bs['current_phase'].map(PHASE_GROUP).fillna('Other / Non-standard')
bs.loc[bs['current_phase'].isna(), 'phase_group'] = 'Unknown'

log(T, 'Standardise phase', f"{bs['current_phase_raw'].nunique()} raw labels → "
                            f"{bs['current_phase'].nunique()} clean phases → {bs['phase_group'].nunique()} groups")

print(bs['phase_group'].value_counts().to_string())
print('\nUnmapped labels grouped as Other / Non-standard:')
print(bs.loc[bs['phase_group'] == 'Other / Non-standard', 'current_phase'].value_counts().to_string())

[budget_schedule] Standardise phase: 60 raw labels → 46 clean phases → 11 groups
phase_group
Pending                 12145
1 Planning               9496
4 Construction           7882
2 Design                 6968
6 Completed              4854
Other / Non-standard     4621
5 Close-out              4451
3 Procurement            3816
On Hold / Inactive       1292
Stopped                   999
Unknown                     1

Unmapped labels grouped as Other / Non-standard:
current_phase
Partner Managed            1389
Requirements Contract      1081
Pass Through Fund           496
Lump Sum                    446
Equipment                   309
Funding Agreement           184
Job Order Contract          167
Consultant Services         166
Holding Code                160
Transferred                  51
It Project                   30
Real Estate Transaction      22
Land Acquisition             20
Ifa                          18
Pre Lease                    16
Fundraising                  16
E

In [ ]:
# CHECK: do bracketed phase labels line up with rows that have no agency schedule (no PID)?
pd.crosstab(bs['phase_in_brackets'], bs['has_schedule'], margins=True)

has_schedule,False,True,All
phase_in_brackets,,,
False,29,26590,26619
True,26432,3474,29906
All,26461,30064,56525


In [ ]:
# 3.5 Borough, numeric sanity checks
if 'borough' in bs.columns:
    bs['borough'] = bs['borough'].str.title()

n_neg = int((bs['total_budget'] < 0).sum())
n_zero = int((bs['total_budget'] == 0).sum())
bs['flag_zero_or_negative_budget'] = bs['total_budget'] <= 0
log(T, 'Budget check', f'{n_neg} negative and {n_zero} zero total_budget rows flagged (kept)', n_neg + n_zero)

bs['flag_spend_exceeds_budget'] = bs['spend_to_date'] > bs['total_budget']
log(T, 'Spend check', 'Rows where spend_to_date > total_budget flagged (kept)', int(bs['flag_spend_exceeds_budget'].sum()))

print(bs[['total_budget', 'spend_to_date', 'spend_to_date_pct']].describe().round(1).to_string())

[budget_schedule] Budget check: 1 negative and 499 zero total_budget rows flagged (kept) (500 rows)
[budget_schedule] Spend check: Rows where spend_to_date > total_budget flagged (kept) (323 rows)
       total_budget  spend_to_date  spend_to_date_pct
count  5.652500e+04   5.652500e+04            56525.0
mean   3.262010e+07   7.765355e+06                0.3
std    1.933334e+08   6.414405e+07                2.3
min   -5.050000e+02   0.000000e+00             -543.6
25%    1.200000e+06   0.000000e+00                0.0
50%    4.598524e+06   1.414926e+05                0.0
75%    1.602720e+07   1.837269e+06                0.6
max    6.798526e+09   2.848666e+09                4.0


In [ ]:
# 3.6 Grain check: how many rows per FMS ID within one snapshot?
rows_per_key = bs.groupby(['reporting_period', 'fms_id']).size()
print('Rows per (reporting_period, fms_id):')
print(rows_per_key.value_counts().sort_index().head(10).to_string())

fms_per_pid = bs.dropna(subset=['pid']).groupby(['reporting_period', 'pid'])['fms_id'].nunique()
print('\nFMS IDs per PID within a snapshot:')
print(fms_per_pid.value_counts().sort_index().head(10).to_string())

Rows per (reporting_period, fms_id):
1     54031
2       625
3        80
4        79
5        10
10        4
11        6
13       10
15        8
16        2

FMS IDs per PID within a snapshot:
fms_id
1     24705
2      1444
3       181
4        54
5        40
6        51
7        17
8        49
9        10
10        3


## 4 · Clean `schedule_history`

In [ ]:
sh = sh_raw.copy()
T = 'schedule_history'

sh.columns = [snake_case(c) for c in sh.columns]
sh = sh.rename(columns={'variance_day': 'variance_days'})
log(T, 'Rename columns', 'snake_case; variance_day → variance_days')

sh = strip_text(sh, T)
sh = drop_exact_dupes(sh, T)

sh['reporting_period'] = yyyymm_to_date(sh['reporting_period'])
sh['pid'] = sh['pid'].astype('Int64')
sh = parse_dates(sh, [c for c in ['completion_date', 'data_date'] if c in sh.columns], T)

sh['completion_date_type'] = sh['completion_date_type'].str.title()
sh['current_phase_raw'] = sh['current_phase']
sh['current_phase'] = sh['current_phase_raw'].apply(normalise_phase)
sh['phase_group'] = sh['current_phase'].map(PHASE_GROUP).fillna('Other / Non-standard')
print(list(sh.columns))

[schedule_history] Rename columns: snake_case; variance_day → variance_days
[schedule_history] Trim text: Stripped whitespace in 7 text columns; blanks set to null
[schedule_history] Exact duplicates: Removed fully duplicated rows (0 rows)
[schedule_history] Date issues: completion_date: 1 unparseable, 0 outside 1980-2050 set to null (1 rows)
[schedule_history] Parse dates: Converted 2 text columns to datetime
['reporting_period', 'managing_agency', 'pid', 'agency_project_name', 'current_phase', 'completion_date', 'completion_date_type', 'variance_days', 'reason_for_forecast_completion_change', 'data_date', 'current_phase_raw', 'phase_group']


### 4.1 Delay reasons → short categories
Near-duplicate labels are merged, and rows without a stated reason are marked explicitly rather than left blank.

In [ ]:
REASON_RULES = [
    ('SCOPE', 'Scope / Design Change'),
    ('PERMIT', 'Permit Approvals'),
    ('BUDGET', 'Budget Constraints'),
    ('SITE/FIELD', 'Unforeseen Site Conditions'),
    ('HAZARDOUS', 'Hazardous Conditions'),
    ('LEGAL', 'Legal Issues'),
    ('UNAVAILABILITY OF PRODUCT', 'Product Unavailability'),
    ('UTILITY', 'Utility Work Scheduling'),
    ('CONTRACTOR DEFAULT', 'Contractor Default'),
    ('NEW TECHNOLOGY', 'Awaiting New Technology'),
    ('GRANT', 'Non-City Grant Approval'),
    ('STATE REQ', 'State Contract Approval'),
]

def categorise_reason(x):
    if pd.isna(x):
        return 'Not stated'
    s = str(x).upper()
    for keyword, label in REASON_RULES:
        if keyword in s:
            return label
    return 'Other'

reason_col = 'reason_for_forecast_completion_change'
sh['delay_reason'] = sh[reason_col].apply(categorise_reason)
log(T, 'Categorise reasons', f'{sh[reason_col].nunique()} raw reasons → {sh["delay_reason"].nunique()} categories')
print(sh['delay_reason'].value_counts().to_string())

[schedule_history] Categorise reasons: 13 raw reasons → 13 categories
delay_reason
Not stated                    20009
Scope / Design Change           734
Permit Approvals                488
Budget Constraints              431
Unforeseen Site Conditions      407
Legal Issues                    147
Product Unavailability          102
Utility Work Scheduling          87
Hazardous Conditions             17
Contractor Default               15
Awaiting New Technology          12
Non-City Grant Approval          10
State Contract Approval           5


In [ ]:
# 4.2 Classify each schedule change
def change_type(v):
    if pd.isna(v): return 'First record'
    if v > 0:      return 'Delay'
    if v < 0:      return 'Acceleration'
    return 'No change'

sh['change_type'] = sh['variance_days'].apply(change_type)
sh = sh.sort_values(['pid', 'reporting_period']).reset_index(drop=True)

print(sh['change_type'].value_counts().to_string())
print('\nVariance (days) for delays:')
print(sh.loc[sh['change_type'] == 'Delay', 'variance_days'].describe().round(0).to_string())

# How often is a reason given when a project slips?
delays = sh[sh['change_type'] == 'Delay']
print(f"\nDelays with a stated reason: {(delays['delay_reason'] != 'Not stated').mean():.1%}")

change_type
No change       8814
First record    8210
Delay           4154
Acceleration    1286

Variance (days) for delays:
count      4154.0
mean        327.0
std        5672.0
min           1.0
25%          88.0
50%         154.0
75%         306.0
max      365362.0

Delays with a stated reason: 59.1%


In [ ]:
# 4.3 Referential check: do schedule_history PIDs exist in budget_schedule?
pids_bs = set(bs['pid'].dropna())
orphans = ~sh['pid'].isin(pids_bs)
log(T, 'Join check', 'PIDs not found in budget_schedule (kept, flagged)', int(orphans.sum()))
sh['flag_pid_not_in_budget'] = orphans

[schedule_history] Join check: PIDs not found in budget_schedule (kept, flagged) (140 rows)


## 5 · Clean `budget_spend_history`

In [ ]:
bsh = bsh_raw.copy()
T = 'budget_spend_history'

bsh.columns = [snake_case(c) for c in bsh.columns]
bsh = strip_text(bsh, T)
bsh = drop_exact_dupes(bsh, T)

bsh['year_month_reported'] = yyyymm_to_date(bsh['year_month_reported'])
bsh['fms_id'] = bsh['fms_id'].str.upper()
bsh = bsh.sort_values(['fms_id', 'year_month_reported']).reset_index(drop=True)
log(T, 'Keys', 'year_month_reported → date; fms_id upper-cased')

print(list(bsh.columns))
print(bsh['year_month_reported'].value_counts().sort_index().to_string())

orphans = ~bsh['fms_id'].isin(set(bs['fms_id']))
log(T, 'Join check', 'FMS IDs not found in budget_schedule (kept, flagged)', int(orphans.sum()))
bsh['flag_fms_not_in_budget'] = orphans

[budget_spend_history] Trim text: Stripped whitespace in 2 text columns; blanks set to null
[budget_spend_history] Exact duplicates: Removed fully duplicated rows (0 rows)
[budget_spend_history] Keys: year_month_reported → date; fms_id upper-cased
['managing_agency', 'fms_id', 'year_month_reported', 'total_budget', 'spend_to_date', 'spend_to_date_pct', 'budget_variance', 'budget_variance_pct']
year_month_reported
2006-09-01     152
2007-01-01      11
2007-04-01      47
2007-09-01      40
2008-01-01      22
2008-04-01       2
2008-11-01      33
2009-01-01       9
2009-04-01       7
2009-09-01      20
2010-04-01       4
2010-09-01      48
2011-02-01      42
2011-05-01       6
2011-09-01      18
2012-01-01      14
2012-04-01      14
2012-10-01      13
2013-01-01      41
2013-04-01      22
2013-10-01      37
2014-02-01       4
2014-05-01      29
2014-10-01     296
2015-02-01      26
2015-05-01      18
2015-09-01     113
2016-01-01      79
2016-04-01      26
2016-10-01      85
2017-01-01   

## 6 · Agency dimension
Agency abbreviations expanded to full names for readable dashboard labels.
*NYRL is not confirmed from the source data; verify it against the NYC Open Data documentation.*

In [ ]:
AGENCY_NAMES = {
    'DPR': 'Department of Parks and Recreation',
    'DDC': 'Department of Design and Construction',
    'DOT': 'Department of Transportation',
    'DEP': 'Department of Environmental Protection',
    'EDC': 'Economic Development Corporation',
    'HHC': 'NYC Health + Hospitals',
    'DCAS': 'Department of Citywide Administrative Services',
    'CUNY': 'City University of New York',
    'NYPD': 'Police Department',
    'FDNY': 'Fire Department',
    'DHS': 'Department of Homeless Services',
    'DCLA': 'Department of Cultural Affairs',
    'DSNY': 'Department of Sanitation',
    'QPL': 'Queens Public Library',
    'DOC': 'Department of Correction',
    'DOHMH': 'Department of Health and Mental Hygiene',
    'NYPL': 'New York Public Library',
    'BPL': 'Brooklyn Public Library',
    'HRA': 'Human Resources Administration',
    'DFTA': 'Department for the Aging',
    'ACS': "Administration for Children's Services",
    'DOE': 'Department of Education',
    'NYRL': 'NYRL (to verify)',
    'HPD': 'Department of Housing Preservation and Development',
    'OTI': 'Office of Technology and Innovation',
}

agencies = sorted(set(bs['managing_agency'].dropna()) | set(sh['managing_agency'].dropna()) | set(bsh['managing_agency'].dropna()))
dim_agency = pd.DataFrame({'agency_code': agencies})
dim_agency['agency_name'] = dim_agency['agency_code'].map(AGENCY_NAMES).fillna(dim_agency['agency_code'])
missing = dim_agency.loc[~dim_agency['agency_code'].isin(AGENCY_NAMES), 'agency_code'].tolist()
print('Agencies without a full name:', missing or 'none')
dim_agency

Agencies without a full name: none


,agency_code,agency_name
0,ACS,Administration for Children's Services
1,BPL,Brooklyn Public Library
2,CUNY,City University of New York
3,DCAS,Department of Citywide Administrative Services
4,DCLA,Department of Cultural Affairs
5,DDC,Department of Design and Construction
6,DEP,Department of Environmental Protection
7,DFTA,Department for the Aging
8,DHS,Department of Homeless Services
9,DOC,Department of Correction


## 7 · Save processed tables

In [ ]:
outputs = {
    'budget_schedule_clean': bs,
    'schedule_history_clean': sh,
    'budget_spend_history_clean': bsh,
    'dim_agency': dim_agency,
    'data_quality_log': pd.DataFrame(dq_log),
}

for name, df in outputs.items():
    path = f'{PROCESSED}/{name}.csv'
    df.to_csv(path, index=False)
    print(f'{name:28s} {df.shape[0]:>7,} rows  {os.path.getsize(path) / 1e6:6.1f} MB')

budget_schedule_clean         56,525 rows    20.3 MB
schedule_history_clean        22,464 rows     3.8 MB
budget_spend_history_clean    53,495 rows     3.4 MB
dim_agency                        25 rows     0.0 MB
data_quality_log                  24 rows     0.0 MB


## 8 · Summary
- **budget_schedule_clean**: all 10 snapshots kept so SQL can compare the earliest and latest budget per FMS ID. *Note: the "original" budget in this analysis is the first snapshot available (May 2023), not necessarily the budget at project approval.*
- **schedule_history_clean**: every completion-date change, classified as Delay / Acceleration / No change, with a short delay-reason category.
- **budget_spend_history_clean**: monthly budget and spend snapshots per FMS ID.
- **dim_agency**: agency codes and full names.
- **data_quality_log**: every cleaning step and the rows it affected.

Next step: load these into PostgreSQL and build the star schema.

In [18]:
dup = bs[bs.duplicated(['reporting_period', 'fms_id'], keep=False)]
same_budget = dup.groupby(['reporting_period', 'fms_id'])[['total_budget', 'spend_to_date']].nunique().max(axis=1).eq(1)
print(f'Repeated (period, FMS ID) groups: {len(same_budget):,}')
print(f'Groups with identical budget on every row: {same_budget.mean():.1%}')

print('\nColumns that differ between repeated rows:')
diff = dup.groupby(['reporting_period', 'fms_id']).nunique().gt(1).sum()
print(diff[diff > 0].sort_values(ascending=False).to_string())

latest = bs[bs['reporting_period'] == bs['reporting_period'].max()]
print(f"\nLatest snapshot total budget - naive sum: ${latest['total_budget'].sum() / 1e9:,.2f}B")
print(f"Latest snapshot total budget - one row per FMS ID: ${latest.drop_duplicates('fms_id')['total_budget'].sum() / 1e9:,.2f}B")

Repeated (period, FMS ID) groups: 834
Groups with identical budget on every row: 63.4%

Columns that differ between repeated rows:
current_phase                            696
current_phase_raw                        696
phase_group                              696
pid                                      535
agency_project_name                      508
agency_project_description               404
managing_agency                          305
total_budget                             305
phase_in_brackets                        296
agency_data_date                         280
spend_to_date                            265
spend_to_date_pct                        264
forecast_completion                      240
forecast_current_phase_end               236
current_phase_start                      191
actual_design_start                      190
has_schedule                             183
actual_design_end                        129
actual_construction_procurement_start    124
budget_line   

In [19]:
def grain_test(df, keys, value_cols=('total_budget', 'spend_to_date')):
    g = df.groupby(keys, dropna=False)
    rows = g.size()
    same = g[list(value_cols)].nunique(dropna=False).max(axis=1).eq(1)
    print(f"{' + '.join(keys)}: {len(rows):,} groups | {(rows > 1).sum():,} repeated | "
          f"budget identical within group: {same.mean():.2%}")

print('budget_schedule:')
for keys in [['reporting_period', 'fms_id'],
             ['reporting_period', 'fms_id', 'managing_agency'],
             ['reporting_period', 'fms_id', 'managing_agency', 'budget_line']]:
    grain_test(bs, keys)

print('\nbudget_spend_history:')
grain_test(bsh, ['year_month_reported', 'fms_id'])
grain_test(bsh, ['year_month_reported', 'fms_id', 'managing_agency'])

dedup = latest.drop_duplicates(['fms_id', 'managing_agency'])
print(f"\nLatest snapshot total, one row per FMS ID + agency: ${dedup['total_budget'].sum() / 1e9:,.2f}B")

budget_schedule:
reporting_period + fms_id: 54,865 groups | 834 repeated | budget identical within group: 99.44%
reporting_period + fms_id + managing_agency: 55,170 groups | 535 repeated | budget identical within group: 100.00%
reporting_period + fms_id + managing_agency + budget_line: 55,170 groups | 535 repeated | budget identical within group: 100.00%

budget_spend_history:
year_month_reported + fms_id: 52,910 groups | 585 repeated | budget identical within group: 98.89%
year_month_reported + fms_id + managing_agency: 53,218 groups | 277 repeated | budget identical within group: 99.48%

Latest snapshot total, one row per FMS ID + agency: $160.34B
